# Week 9 · Topic 4 — Implementing a Simplified Transformer Block
### Demo Notebook: Assembling and Stacking a Complete Transformer Block (Pure Python + NumPy)

**Goal:** bring every piece built this week together into one complete, working Transformer
block — then stack it, exactly as described in Topic 3's encoder.

**A note on style:** this notebook reuses `multi_head_attention()` from Topic 2 completely
unchanged — including its `@` projections, which Topic 1 already proved are equivalent to
writing the math out by hand. The genuinely new operation this topic introduces is **layer
normalisation**, which needs a mean and a variance computed *per token*. Rather than NumPy's
`np.mean(X, axis=-1, keepdims=True)` (which requires understanding what "axis" and "keepdims"
mean), we compute it with a plain loop, one token at a time — same result, nothing hidden.

Five parts:
1. **Recap** — bring in Topics 1–2's functions, unchanged
2. **Residual connections + layer normalisation** — the wrapper pattern used around every sublayer
3. **The feed-forward network** — new this topic: linear → ReLU → linear
4. **Assembling one full block** — attention sublayer + feed-forward sublayer, each wrapped
5. **Stacking blocks** — building the deep encoder from Topic 3

Running example (same as Topics 1–2): **English** `"I am going to the market"` /
**Yoruba** `"Mo n lọ si ọja"`.

In [ ]:
import numpy as np
import math

np.set_printoptions(precision=3, suppress=True)
print("NumPy and math ready.")


NumPy and math ready.


## Part 1 — Recap: Reusing Topics 1–2's Functions

Everything below is copied unchanged from Topics 1 and 2: the from-scratch softmax, the
attention mechanism, head-splitting, head-combining, multi-head attention, and positional
encoding. None of it needs to change to support this topic's new content.

In [ ]:
# --- Reused from Topic 1, unchanged ---
def softmax(scores):
    max_score = max(scores)
    exp_scores = [math.exp(s - max_score) for s in scores]
    total = sum(exp_scores)
    return [e / total for e in exp_scores]

def softmax_rows(matrix):
    return np.array([softmax(list(row)) for row in matrix])

def attention_from_qkv(Q, K, V):
    d_k = Q.shape[-1]
    raw_scores = Q @ K.T
    scaled_scores = raw_scores / math.sqrt(d_k)
    weights = softmax_rows(scaled_scores)
    output = weights @ V
    return output, weights

print("Topic 1 functions reused.")


Topic 1 functions reused.


In [ ]:
# --- Reused from Topic 2, unchanged ---
def split_into_heads(matrix, num_heads):
    head_dim = matrix.shape[1] // num_heads
    heads = []
    for h in range(num_heads):
        start = h * head_dim
        end = start + head_dim
        one_head = np.array([row[start:end] for row in matrix])
        heads.append(one_head)
    return heads

def concatenate_heads(head_outputs):
    num_tokens = head_outputs[0].shape[0]
    combined = []
    for i in range(num_tokens):
        row = []
        for head_output in head_outputs:
            row.extend(list(head_output[i]))
        combined.append(row)
    return np.array(combined)

def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    Q_heads = split_into_heads(Q, num_heads)
    K_heads = split_into_heads(K, num_heads)
    V_heads = split_into_heads(V, num_heads)
    outputs, all_weights = [], []
    for h in range(num_heads):
        out_h, w_h = attention_from_qkv(Q_heads[h], K_heads[h], V_heads[h])
        outputs.append(out_h)
        all_weights.append(w_h)
    combined = concatenate_heads(outputs)
    output = combined @ W_O
    return output, all_weights

def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for dim in range(d_model):
            angle = pos / (10000 ** (2 * (dim // 2) / d_model))
            if dim % 2 == 0:
                pe[pos][dim] = math.sin(angle)
            else:
                pe[pos][dim] = math.cos(angle)
    return pe

print("Topic 2 functions reused: multi_head_attention(), positional_encoding()")


Topic 2 functions reused: multi_head_attention(), positional_encoding()


In [ ]:
# The running example, set up exactly as in Topic 2: embeddings + positional encoding.
tokens_full = ["I", "am", "going", "to", "the", "market"]
yoruba_reference = "Mo n lọ si ọja"

d_model = 8
np.random.seed(7)
embeddings_full = {t: np.round(np.random.randn(d_model), 2) for t in tokens_full}
X_full = np.array([embeddings_full[t] for t in tokens_full])

PE = positional_encoding(len(tokens_full), d_model)
X_pos = X_full + PE   # position-aware input, ready to enter a Transformer block

num_heads = 2
np.random.seed(11)
W_Q = np.round(np.random.randn(d_model, d_model) * 0.3, 2)
W_K = np.round(np.random.randn(d_model, d_model) * 0.3, 2)
W_V = np.round(np.random.randn(d_model, d_model) * 0.3, 2)
W_O = np.round(np.random.randn(d_model, d_model) * 0.3, 2)

print("English tokens:", tokens_full)
print("Yoruba translation (reference only):", yoruba_reference)
print("X_pos shape:", X_pos.shape, "(6 tokens x 8 dimensions, position-aware)")


English tokens: ['I', 'am', 'going', 'to', 'the', 'market']
Yoruba translation (reference only): Mo n lọ si ọja
X_pos shape: (6, 8) (6 tokens x 8 dimensions, position-aware)


## Part 2 — Residual Connections + Layer Normalisation

**The pattern (Slide 5):** every sublayer in a Transformer block is wrapped the same way:

**`output = LayerNorm(sublayer(x) + x)`**

We'll use this exact wrapper twice in this notebook — once around attention, once around the
feed-forward network. Let's build the `layer_norm()` piece first, since it's brand new.

In [ ]:
# Layer normalisation, written as a plain loop over tokens (rows).
# For each token's vector: subtract its own mean, divide by its own standard
# deviation. No np.mean(axis=-1, keepdims=True) -- we compute mean and
# variance by hand, one row at a time, so nothing about "which axis" is left implicit.
def layer_norm(X, eps=1e-6):
    normalized_rows = []
    for row in X:
        mean = sum(row) / len(row)
        variance = sum((value - mean) ** 2 for value in row) / len(row)
        std = math.sqrt(variance + eps)
        normalized_row = [(value - mean) / std for value in row]
        normalized_rows.append(normalized_row)
    return np.array(normalized_rows)

# Quick check: normalise our position-aware input and confirm each row now has
# (approximately) mean 0 and standard deviation 1.
test_normed = layer_norm(X_pos)
print("Row 0 after layer norm:", test_normed[0])
print("Row 0 mean (should be ~0):", round(test_normed[0].mean(), 6))
print("Row 0 std  (should be ~1):", round(test_normed[0].std(), 6))


Row 0 after layer norm: [ 1.489  0.16  -0.412  1.169 -1.352  0.699 -0.447 -1.306]
Row 0 mean (should be ~0): 0.0
Row 0 std  (should be ~1): 0.999999


**What this confirms:** no matter how large or small a token's values were beforehand,
layer normalisation rescales them into the same stable range — exactly Slide 6's point about
keeping values well-behaved as they pass through many stacked layers.

## Part 3 — The Feed-Forward Network (New This Topic)

**Slide 7:** a small two-layer network applied *independently* to each token's representation —
**Linear → ReLU → Linear**.

In [ ]:
# Set up the feed-forward network's weights.
# d_ff is the "expanded" size in the middle of the network -- larger than
# d_model, giving the network more room to work before compressing back down.
d_ff = 16
np.random.seed(21)
W1 = np.round(np.random.randn(d_model, d_ff) * 0.3, 2)   # expands: d_model -> d_ff
b1 = np.zeros(d_ff)
W2 = np.round(np.random.randn(d_ff, d_model) * 0.3, 2)   # compresses: d_ff -> d_model
b2 = np.zeros(d_model)

print("W1 shape (expand):", W1.shape)
print("W2 shape (compress):", W2.shape)


W1 shape (expand): (8, 16)
W2 shape (compress): (16, 8)


In [ ]:
# The feed-forward network itself: Linear -> ReLU -> Linear.
# @ is used here for the same reason as Q/K/V projection -- it IS the linear
# layer, not a shortcut hiding it. np.maximum(0, x) is ReLU: keep positive
# values, zero out the rest -- a simple elementwise comparison, not a slicing trick.
def feed_forward(X, W1, b1, W2, b2):
    hidden = X @ W1 + b1          # Linear layer 1: expand
    hidden_relu = np.maximum(0, hidden)   # ReLU: zero out negative values
    output = hidden_relu @ W2 + b2         # Linear layer 2: compress back down
    return output

ff_test_output = feed_forward(X_pos, W1, b1, W2, b2)
print("Feed-forward output shape:", ff_test_output.shape, "(back to 6 tokens x 8 dimensions)")
print(ff_test_output)


Feed-forward output shape: (6, 8) (back to 6 tokens x 8 dimensions)
[[-0.191 -0.288  0.055 -0.155 -0.249 -0.24   0.28   0.656]
 [ 0.503 -0.677  0.391 -0.676  0.041 -0.88  -0.424  1.189]
 [-0.077 -1.146  0.298 -0.209  0.071 -1.008 -0.738  0.946]
 [-0.681 -0.067  0.672 -0.076 -0.28  -0.986 -0.813  0.997]
 [-0.319  0.601  0.628 -1.179 -0.45   0.445 -1.507  0.117]
 [ 1.901 -0.479 -0.012 -1.162  0.531  0.13  -0.376 -0.62 ]]


**Check:** even though the network expands to 16 dimensions in the middle, the output shape
matches the input shape (6 × 8) — the expansion is purely internal, giving the network more room
to compute, but the sublayer's contract with the rest of the block stays the same size in, same
size out.

## Part 4 — Assembling One Full Transformer Block

**Slide 4's seven steps, now in code:**
1. Take the input
2. Run multi-head self-attention
3. Add the attention output back to the input (residual)
4. Layer normalise
5. Run the feed-forward network
6. Add the feed-forward output back to *its* input (residual)
7. Layer normalise again

In [ ]:
def transformer_block(X, W_Q, W_K, W_V, W_O, num_heads, W1, b1, W2, b2):
    """
    One complete Transformer encoder block:
      LayerNorm(attention(X) + X)  ->  LayerNorm(feed_forward(...) + ...)

    Returns:
        output : (seq_len, d_model) -- same shape as the input
        attn_weights : the attention weight matrices from this block's attention sublayer
    """
    # Sublayer 1: multi-head self-attention, wrapped in residual + norm
    attn_output, attn_weights = multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads)
    X_after_attention = layer_norm(X + attn_output)

    # Sublayer 2: feed-forward network, wrapped in residual + norm
    ff_output = feed_forward(X_after_attention, W1, b1, W2, b2)
    X_after_ff = layer_norm(X_after_attention + ff_output)

    return X_after_ff, attn_weights

print("transformer_block() is defined and ready to run.")


transformer_block() is defined and ready to run.


In [ ]:
# Run one full block on our position-aware running example.
block_output, block_attn_weights = transformer_block(
    X_pos, W_Q, W_K, W_V, W_O, num_heads, W1, b1, W2, b2
)

print("Output of one Transformer block:")
for tok, vec in zip(tokens_full, block_output):
    print(f"  {tok:8s} -> {vec}")

print("\nOutput shape:", block_output.shape, "(matches input shape: 6 tokens x 8 dimensions)")


Output of one Transformer block:
  I        -> [-0.1   -0.216 -0.158  0.731 -1.974  1.461  0.94  -0.684]
  am       -> [ 1.829  0.804 -1.074 -1.245 -0.088  0.845 -0.469 -0.601]
  going    -> [-0.017 -0.792 -0.543 -1.276  1.212  0.584 -0.867  1.699]
  to       -> [-0.681 -1.636  0.159 -0.519  0.733  0.443 -0.414  1.913]
  the      -> [-1.744 -0.156  0.07  -0.231 -1.126  1.443  0.811  0.932]
  market   -> [ 0.759 -0.166  0.08  -1.194 -1.374 -0.296  0.203  1.987]

Output shape: (6, 8) (matches input shape: 6 tokens x 8 dimensions)


**Why the matching shape matters (Slide 9):** because this block's output has the exact
same shape as its input, the output of one block can be fed straight into another block as its
input. That's what makes stacking possible — which is exactly what Topic 3's encoder does.

## Part 5 — Stacking Blocks

**Slide 10:** a real encoder stacks several blocks — each with its own independently learned
weights — one after another. Here we'll stack 2 blocks, each with its own random weights
(standing in for what a model would actually learn).

In [ ]:
# Generate a fresh, independent set of weights for one block.
def random_block_weights(d_model, d_ff, seed):
    rng = np.random.RandomState(seed)
    W_Q = np.round(rng.randn(d_model, d_model) * 0.3, 2)
    W_K = np.round(rng.randn(d_model, d_model) * 0.3, 2)
    W_V = np.round(rng.randn(d_model, d_model) * 0.3, 2)
    W_O = np.round(rng.randn(d_model, d_model) * 0.3, 2)
    W1 = np.round(rng.randn(d_model, d_ff) * 0.3, 2)
    b1 = np.zeros(d_ff)
    W2 = np.round(rng.randn(d_ff, d_model) * 0.3, 2)
    b2 = np.zeros(d_model)
    return W_Q, W_K, W_V, W_O, W1, b1, W2, b2

num_blocks = 2
block_params = [random_block_weights(d_model, d_ff, seed=100 + i) for i in range(num_blocks)]
print(f"Generated independent weights for {num_blocks} blocks.")


Generated independent weights for 2 blocks.


In [ ]:
# Feed the input through each block in turn -- each block's output becomes
# the next block's input, exactly like the encoder stack in Topic 3.
X_stacked = X_pos
for i, params in enumerate(block_params):
    W_Q_i, W_K_i, W_V_i, W_O_i, W1_i, b1_i, W2_i, b2_i = params
    X_stacked, _ = transformer_block(
        X_stacked, W_Q_i, W_K_i, W_V_i, W_O_i, num_heads, W1_i, b1_i, W2_i, b2_i
    )
    print(f"After block {i + 1}: shape {X_stacked.shape}")

print("\nFinal output after the full stack:")
for tok, vec in zip(tokens_full, X_stacked):
    print(f"  {tok:8s} -> {vec}")


After block 1: shape (6, 8)
After block 2: shape (6, 8)

Final output after the full stack:
  I        -> [ 1.88  -1.442  0.954 -0.777 -0.511  0.6   -0.392 -0.313]
  am       -> [ 0.138  1.274 -1.465  0.057  1.208 -1.307 -0.671  0.768]
  going    -> [ 0.006 -0.238 -0.4   -1.267  0.559 -0.846 -0.082  2.268]
  to       -> [-0.013 -1.446  0.602 -0.933  0.846 -0.711 -0.194  1.849]
  the      -> [ 0.407 -0.377  1.483 -0.911 -1.303  1.278  0.475 -1.052]
  market   -> [ 1.056 -0.768  0.674 -0.81  -1.282 -0.857  0.349  1.638]


**What we just built:** starting from `"I am going to the market"`, we tokenized it,
added positional encoding, and passed it through 2 stacked Transformer blocks — each one
running multi-head attention, a residual connection, layer normalisation, a feed-forward
network, another residual connection, and another layer normalisation. The shape stayed
`(6, 8)` throughout, which is exactly what lets a real encoder stack this same block 6, 12, or
more times.

## Wrap-Up

In this notebook we:

- Reused every function from Topics 1–2 completely unchanged — multi-head attention,
  positional encoding, and the underlying attention mechanism
- Built `layer_norm()` from scratch as a plain per-token loop, avoiding NumPy's
  `axis=`/`keepdims=` broadcasting
- Built `feed_forward()` — a two-layer network (Linear → ReLU → Linear) — the one genuinely
  new sublayer this topic introduces
- Assembled `transformer_block()`: attention wrapped in residual + norm, followed by
  feed-forward wrapped in residual + norm — Slide 4's seven steps, in code
- Confirmed the block's output shape matches its input shape, and used that fact to stack
  2 independent blocks, feeding one block's output into the next

**Up next: the Module Demo.** All four topics — attention, multi-head attention with
positional encoding, the architecture behind encoder/decoder stacks, and this complete,
stackable Transformer block — come together into one end-to-end pipeline, framed around
building a mini translation engine for NaijaLingo.